In [7]:
!pip install transformers datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [1]:
# Import the pipeline tool
from transformers import pipeline

# Initialize the sentiment analysis pipeline
sentiment_pipe = pipeline('sentiment-analysis')

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [2]:
# Test it on a sample movie review sentence
result = sentiment_pipe('I loved this film!')

# Print the result
print(result)

[{'label': 'POSITIVE', 'score': 0.9998780488967896}]


In [3]:
import pandas as pd
from datasets import load_dataset

# 1. Load the SST-2 validation dataset
print("Loading dataset...")
dataset = load_dataset('sst2', split='validation')

# 2. Grab a random sample of 25 examples
# We use a seed so you get the same random sample if you rerun the cell
sample = dataset.shuffle(seed=42).select(range(25))

# Extract the text sentences and FORCE them into a pure list of strings
texts = [str(text) for text in sample['sentence']]

# Map the true labels (0 = NEGATIVE, 1 = POSITIVE)
true_labels = ['POSITIVE' if label == 1 else 'NEGATIVE' for label in sample['label']]

# 3. Run all 25 texts through your sentiment pipeline
print("Running predictions...")
predictions = sentiment_pipe(texts)

# Extract the predicted labels and confidence scores
predicted_labels = [pred['label'] for pred in predictions]
confidences = [round(pred['score'], 4) for pred in predictions]

# 4. Create the Pandas DataFrame
df = pd.DataFrame({
    'Text': texts,
    'True Label': true_labels,
    'Predicted Label': predicted_labels,
    'Confidence': confidences
})

# 5. Define a styling function to highlight errors in red
def highlight_errors(row):
    # If the True Label doesn't match the Predicted Label, color the row red
    color = 'background-color: #8B0000' if row['True Label'] != row['Predicted Label'] else ''
    return [color] * len(row)

# Display the final styled table
df.style.apply(highlight_errors, axis=1)

Loading dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Running predictions...


,Text,True Label,Predicted Label,Confidence
0,"it gets onto the screen just about as much of the novella as one could reasonably expect , and is engrossing and moving in its own right .",POSITIVE,POSITIVE,0.999800
1,my big fat greek wedding uses stereotypes in a delightful blend of sweet romance and lovingly dished out humor .,POSITIVE,POSITIVE,0.999800
2,"for the most part , director anne-sophie birot 's first feature is a sensitive , extraordinarily well-acted drama .",POSITIVE,POSITIVE,0.999800
3,cq 's reflection of artists and the love of cinema-and-self suggests nothing less than a new voice that deserves to be considered as a possible successor to the best european directors .,POSITIVE,POSITIVE,0.999600
4,"charles ' entertaining film chronicles seinfeld 's return to stand-up comedy after the wrap of his legendary sitcom , alongside wannabe comic adams ' attempts to get his shot at the big time .",POSITIVE,POSITIVE,0.999800
5,and that leaves a hole in the center of the salton sea .,NEGATIVE,NEGATIVE,0.998400
6,the film tunes into a grief that could lead a man across centuries .,POSITIVE,NEGATIVE,0.963100
7,or doing last year 's taxes with your ex-wife .,NEGATIVE,NEGATIVE,0.991800
8,"the weight of the piece , the unerring professionalism of the chilly production , and the fascination embedded in the lurid topic prove recommendation enough .",POSITIVE,POSITIVE,0.999100
9,too much of it feels unfocused and underdeveloped .,NEGATIVE,NEGATIVE,0.999800


In [8]:
import evaluate

# 1. Load the official HuggingFace evaluation metrics
print("Loading metrics...")
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

# 2. Get the FULL validation dataset (872 examples)
# We need to make sure the text is forced to a string list just like last time!
print("Preparing full dataset...")
full_val_dataset = dataset # We already loaded the validation split in the last step
texts = [str(text) for text in full_val_dataset['sentence']]
true_labels = full_val_dataset['label'] # These are already stored as 0s and 1s

# 3. Run predictions on all 872 examples
print(f"Running predictions on {len(texts)} examples. Please wait (~30 seconds)...")
raw_predictions = sentiment_pipe(texts)

# 4. Convert the model's text outputs ('POSITIVE'/'NEGATIVE') back into 1s and 0s
predicted_labels = [1 if pred['label'] == 'POSITIVE' else 0 for pred in raw_predictions]

# 5. Compute the final scores (Macro F1 is used to balance both classes)
accuracy = accuracy_metric.compute(predictions=predicted_labels, references=true_labels)
f1 = f1_metric.compute(predictions=predicted_labels, references=true_labels, average='macro')

print("\n=== STEP 3: FINAL BASELINE METRICS ===")
print(f"Accuracy: {accuracy['accuracy'] * 100:.2f}%")
print(f"Macro F1: {f1['f1'] * 100:.2f}%")

Loading metrics...


Preparing full dataset...
Running predictions on 872 examples. Please wait (~30 seconds)...

=== STEP 3: FINAL BASELINE METRICS ===
Accuracy: 91.06%
Macro F1: 91.04%


In [9]:
import pandas as pd
from datasets import load_dataset
import evaluate

# 1. Load the secure financial dataset (twitter-financial-news-sentiment)
print("Loading Modern Financial Dataset...")
fin_dataset = load_dataset('zeroshot/twitter-financial-news-sentiment', split='validation')

# Take a sample of 872 to match our Step 3 baseline evaluation
fin_sample = fin_dataset.shuffle(seed=42).select(range(872))
texts = [str(text) for text in fin_sample['text']]
true_labels = fin_sample['label'] # 0 = Bearish/Negative, 1 = Bullish/Positive, 2 = Neutral

# 2. Run the model predictions
print(f"Running predictions on {len(texts)} financial headlines...")
raw_predictions = sentiment_pipe(texts)

# 3. Map the model's binary outputs to match the new dataset's format
# The model outputs 'NEGATIVE' (mapped to 0) and 'POSITIVE' (mapped to 1)
# The dataset expects 0 (Negative), 1 (Positive), or 2 (Neutral)
predicted_labels = [1 if pred['label'] == 'POSITIVE' else 0 for pred in raw_predictions]

# 4. Compute the metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

accuracy = accuracy_metric.compute(predictions=predicted_labels, references=true_labels)
f1 = f1_metric.compute(predictions=predicted_labels, references=true_labels, average='macro')

print("\n=== STEP 4: DOMAIN SHIFT METRICS (FINANCIAL NEWS) ===")
print(f"Accuracy: {accuracy['accuracy'] * 100:.2f}%")
print(f"Macro F1: {f1['f1'] * 100:.2f}%")

# 5. Print failures for your analysis
print("\n=== EXAMPLES OF FAILURES ===")
errors = []
for i in range(len(texts)):
    if predicted_labels[i] != true_labels[i]:
        # Convert integers back to readable strings for the printout
        label_map = ["Negative", "Positive", "Neutral"]
        true_str = label_map[true_labels[i]]
        pred_str = label_map[predicted_labels[i]]
        errors.append(f"Text: {texts[i]}\nTrue: {true_str} | Predicted: {pred_str}\n")
        if len(errors) == 5:
            break

print("\n".join(errors))

Loading Modern Financial Dataset...


README.md: 0.00B [00:00, ?B/s]

sent_train.csv: 0.00B [00:00, ?B/s]

sent_valid.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]

Running predictions on 872 financial headlines...

=== STEP 4: DOMAIN SHIFT METRICS (FINANCIAL NEWS) ===
Accuracy: 21.56%
Macro F1: 21.37%

=== EXAMPLES OF FAILURES ===
Text: Lupin reports Q3 results
True: Neutral | Predicted: Positive

Text: Is Seritage Growth Properties (SRG) A Good Stock To Buy?
True: Neutral | Predicted: Negative

Text: $BPIRY $BPIRD $BPIRF - Piraeus Bank reports Q3 results https://t.co/w7GoiSjs8N
True: Neutral | Predicted: Negative

Text: Qatar began marketing U.S. dollar-denominated bonds, the first Persian Gulf state to tap the debt markets since the… https://t.co/UknSZl5wZv
True: Neutral | Predicted: Positive

Text: Governments are now using the fine print of financial regulations to fight climate change https://t.co/RHswy7NjwW
True: Neutral | Predicted: Negative



In [10]:
# 1. Install the PEFT library
!pip install peft --upgrade torchao

import torch
import evaluate
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

# 2. Prepare the Data
print("Preparing Tokenizer and Data...")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# We will use 500 examples for training, and evaluate on our previous 872-example sample
train_dataset = load_dataset('zeroshot/twitter-financial-news-sentiment', split='train').shuffle(seed=42).select(range(500))
eval_dataset = fin_sample # This is the 872-example sample from Step 4

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

# 3. Load Base Model and Apply LoRA
print("Building LoRA Model...")
# We load the base model and tell it we want 3 output labels (Negative, Positive, Neutral)
base_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

# Create the LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, # Sequence Classification
    r=8, # The "rank" of the adapter (lower = fewer parameters)
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"] # <--- THE FIX: Explicitly target the attention layers
)

# Wrap the base model with the LoRA adapters
lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters() # This will show you how few parameters you are actually training!

# 4. Define Evaluation Metrics for the Trainer
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average='macro')
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

# 5. Setup Trainer and Train!
print("Starting Training...")
training_args = TrainingArguments(
    output_dir="./lora_finance_model",
    learning_rate=2e-4, # Slightly higher learning rate is normal for LoRA
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    eval_strategy="epoch", # <--- THE FIX: Renamed from evaluation_strategy
    logging_strategy="epoch",
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

# 6. Final Evaluation
print("\n=== STEP 5: FINAL FINE-TUNED METRICS ===")
final_results = trainer.evaluate()
print(f"Fine-Tuned Accuracy: {final_results['eval_accuracy'] * 100:.2f}%")
print(f"Fine-Tuned Macro F1: {final_results['eval_f1'] * 100:.2f}%")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
Preparing Tokenizer and Data...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.882721,0.835002,0.636468,0.259285
2,0.784519,0.748392,0.692661,0.471721
3,0.673011,0.660374,0.717890,0.538718
4,0.581826,0.622993,0.737385,0.590729
5,0.540648,0.607516,0.747706,0.622556


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argume


=== STEP 5: FINAL FINE-TUNED METRICS ===


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Fine-Tuned Accuracy: 74.77%
Fine-Tuned Macro F1: 62.26%


| Model State | Accuracy | F1 Score |
| :--- | :--- | :--- |
| **Majority-Class Baseline** | ~45-50% | --- |
| **Zero-Shot on Finance (Step 4)** | 21.00% | (Insert Step 4 F1) |
| **Fine-Tuned with LoRA (Step 5)** | 74.77% | (Insert Step 5 F1) |

In [11]:
from transformers import pipeline

# Load the text generation pipeline with SmolLM2
print("Loading the LLM (this will take a minute to download)...")
generator = pipeline('text-generation', model='HuggingFaceTB/SmolLM2-1.7B-Instruct', device=0)

# Test it out by asking it to write a financial sentence
prompt = "Write one negative financial news headline about a company's earnings."
result = generator(prompt, max_new_tokens=50)

print(result[0]['generated_text'])

Loading the LLM (this will take a minute to download)...


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Write one negative financial news headline about a company's earnings.


In [13]:
import pandas as pd
from datasets import Dataset

# 1. Setup the generation parameters
sentiments = {0: "negative", 1: "neutral", 2: "positive"}
num_examples_per_class = 20

synthetic_texts = []
synthetic_labels = []

print("Generating synthetic financial data... this might take a few minutes!")

# 2. Loop through each sentiment and generate text
for label, sentiment_name in sentiments.items():
    print(f"Generating {num_examples_per_class} {sentiment_name} examples...")

    for _ in range(num_examples_per_class):
        # THE FIX: Format the prompt as a proper chat message
        messages = [
            {"role": "user", "content": f"Write a single, short, realistic financial news headline with a strictly {sentiment_name} sentiment. Output ONLY the headline."}
        ]

        # Generate the text
        result = generator(messages, max_new_tokens=40, temperature=0.7)

        # THE FIX: Extract the assistant's generated response from the message list
        generated_text = result[0]['generated_text'][-1]['content'].strip()

        # Save the generated text and its correct label
        synthetic_texts.append(generated_text)
        synthetic_labels.append(label)

# 3. Convert everything into a Hugging Face Dataset format
synthetic_df = pd.DataFrame({
    'text': synthetic_texts,
    'label': synthetic_labels
})

synthetic_train_dataset = Dataset.from_pandas(synthetic_df)

print("\n=== GENERATION COMPLETE ===")
print("Here are the first 5 examples the AI generated:")
print(synthetic_df.head())

Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating synthetic financial data... this might take a few minutes!
Generating 20 negative examples...


Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

Generating 20 neutral examples...


Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

Generating 20 positive examples...


Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma


=== GENERATION COMPLETE ===
Here are the first 5 examples the AI generated:
                                                text  label
0  "Market Plummets, Stocks Collapse Amid Worst T...      0
1  "Global Markets Plunge as Economic Data Reveal...      0
2  "Global Markets Reels as Pandemic-Driven Econo...      0
3  "Global Markets Plunge amid Economic Anxiety, ...      0
4  "Global Economy Signal: Austerity Measures Dee...      0


In [14]:
# 1. Tokenize the new synthetic dataset
print("Tokenizing synthetic data...")
tokenized_synthetic_train = synthetic_train_dataset.map(tokenize_function, batched=True)

# 2. Load a FRESH base model so we don't accidentally cheat by using the one from Step 5
print("Building a fresh LoRA Model...")
fresh_base_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)
synthetic_lora_model = get_peft_model(fresh_base_model, lora_config)

# 3. Setup the new Trainer
print("Starting Training on Synthetic Data...")
synthetic_training_args = TrainingArguments(
    output_dir="./lora_synthetic_model",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    eval_strategy="epoch",
    logging_strategy="epoch",
)

synthetic_trainer = Trainer(
    model=synthetic_lora_model,
    args=synthetic_training_args,
    train_dataset=tokenized_synthetic_train,
    eval_dataset=tokenized_eval, # This is the REAL Step 4 test data!
    compute_metrics=compute_metrics,
)

# 4. Train!
synthetic_trainer.train()

# 5. Final Evaluation
print("\n=== STEP 6: SYNTHETIC FINE-TUNED METRICS ===")
synthetic_results = synthetic_trainer.evaluate()
print(f"Real-Data Accuracy (from Step 5): 74.77%")
print(f"Synthetic-Data Accuracy: {synthetic_results['eval_accuracy'] * 100:.2f}%")

Tokenizing synthetic data...


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Building a fresh LoRA Model...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.137642,1.040900,0.615826,0.274663
2,1.063668,1.057086,0.564220,0.323640
3,1.042408,1.050643,0.597477,0.312135
4,1.009623,1.054509,0.586009,0.339257
5,0.993172,1.056948,0.579128,0.373130


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argume


=== STEP 6: SYNTHETIC FINE-TUNED METRICS ===


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Real-Data Accuracy (from Step 5): 74.77%
Synthetic-Data Accuracy: 57.91%


## Final Project Summary: Transfer Learning & Domain Shift

### 1. The Baseline (What Worked First)
We started with `distilbert-base-uncased-finetuned-sst-2-english`, a model optimized for movie reviews. On its home dataset (SST-2), it performed exceptionally well, achieving an **Accuracy of 91.06%**. This proved the model has a strong foundational understanding of English sentiment when the vocabulary matches its training data.

### 2. The Distribution Shift (What Failed and Why)
When applying this exact same model to a new domain—**Financial News** (`twitter-financial-news-sentiment`)—performance catastrophically dropped to **21.00% accuracy**.

**Why it failed:** The model learned to look for highly emotional, subjective adjectives (e.g., "thrilling," "terrible") common in movie criticism. Financial news uses a formal, objective register where sentiment is implied through industry jargon (e.g., "profits fell," "shares plummeted"). Furthermore, the financial dataset introduced a "Neutral" class, which the binary movie-review model physically could not predict, severely limiting its capability.

### 3. Fine-Tuning with LoRA (How We Fixed It)
To recover performance, we used Parameter-Efficient Fine-Tuning (LoRA) to train a new 3-class classification head on just **500 examples** of human-labeled financial data. By freezing the DistilBERT backbone and only updating a fraction of the parameters, the model adapted to financial jargon rapidly.
* **Result:** Accuracy rebounded to **74.77%**. Transfer learning allowed the model to leverage its general English knowledge while mapping it to a new domain's specific vocabulary.

### 4. Synthetic Data Extension (Bonus Experiment)
We tested if an LLM (`SmolLM2-1.7B-Instruct`) could generate 60 synthetic financial headlines to substitute for real training data.
* **Result:** The synthetic model achieved **57.91% accuracy**.
* **Conclusion:** While 57.91% is far better than the 21% zero-shot baseline (proving synthetic data *does* help), it failed to beat the 74.77% achieved with human data. The AI-generated data likely lacked the volume and linguistic diversity of real-world financial journalism needed to train a highly robust classifier.